# English → Kannada Neural Machine Translation — Kaggle Edition

This notebook is adapted for **Kaggle Notebooks** (no Google Colab imports or Drive mounting).

## Before running
1. Create a Kaggle Dataset containing `pib_2017_2020_en_kn.zip`.
2. Open this notebook in Kaggle and click **Add Input** to attach that dataset.
3. In Notebook **Settings**, select a GPU accelerator (T4 if available) and enable Internet.
4. Run cells sequentially.
5. All generated files are saved to `/kaggle/working/English_Kannada_MT`. At the end, use **Save Version** and enable saving notebook outputs, or download the output folder as a ZIP.

## Experiments
- Basic encoder–decoder Transformer trained from scratch.
- Fine-tuned mT5-small.
- Zero-shot prompting with LLaMA 3.2 1B Instruct (gated access may apply; Kannada is not officially listed as a supported language).
- Zero-shot prompting with Gemma 3 1B IT (Hugging Face access approval may apply).

**Important:** This is a government/news-domain corpus with some noisy or misaligned pairs. Results are domain-specific. Do not report a model's metrics if its experiment did not run successfully. The LLM experiments are optional and may be skipped if model access or GPU memory is unavailable.

## 1. Environment and configuration

In [ ]:
!pip -q install transformers datasets accelerate sentencepiece sacrebleu evaluate pandas numpy matplotlib scikit-learn bitsandbytes

In [ ]:
import os, time, json, math, gc, zipfile, shutil, random, re, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sacrebleu.metrics import BLEU, CHRF

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Enable it in Kaggle Notebook Settings.")

# Kaggle input datasets are read-only; all generated files go to /kaggle/working.
PROJECT_DIR = "/kaggle/working/English_Kannada_MT"
os.makedirs(PROJECT_DIR, exist_ok=True)
for p in ["data/raw", "data/processed", "models", "results/plots", "results/predictions"]:
    os.makedirs(os.path.join(PROJECT_DIR, p), exist_ok=True)

# Experiment switches. Turn off any model that you cannot run within your GPU/time budget.
RUN_BASIC_TRANSFORMER = True
RUN_MT5_FINETUNING = True
RUN_LLAMA_PROMPTING = True
RUN_GEMMA_PROMPTING = True

BASIC_TRAIN_LIMIT = 40282
BASIC_EPOCHS = 15
MT5_TRAIN_LIMIT = 5000
MT5_EPOCHS = 1
LLM_EVAL_LIMIT = 100
EVAL_LIMIT = 500

print("Working directory:", PROJECT_DIR)
print("Kaggle input directories:", glob.glob("/kaggle/input/*"))

## 2. Upload and inspect the Anuvaad dataset

In [ ]:
# Attach the Kaggle Dataset containing pib_2017_2020_en_kn.zip:
# Notebook sidebar -> Add Input -> select your uploaded dataset.
# Kaggle inputs are read-only, so we copy the ZIP into /kaggle/working.
matches = glob.glob("/kaggle/input/**/pib_2017_2020_en_kn.zip", recursive=True)
if not matches:
    raise FileNotFoundError(
        "Could not find pib_2017_2020_en_kn.zip. Upload it as a Kaggle Dataset, "
        "attach that dataset using Add Input, then rerun this cell."
    )

source_zip = matches[0]
zip_path = os.path.join(PROJECT_DIR, "data/raw/pib_en_kn.zip")
shutil.copy2(source_zip, zip_path)
print("Using dataset:", source_zip)

with zipfile.ZipFile(zip_path) as z:
    bad_file = z.testzip()
    print("ZIP integrity:", bad_file)
    print("Archive files:", z.namelist()[:10])
    z.extractall(os.path.join(PROJECT_DIR, "data/raw"))

raw_dir = os.path.join(PROJECT_DIR, "data/raw")
english_candidates = list(Path(raw_dir).rglob("pib_train.en"))
kannada_candidates = list(Path(raw_dir).rglob("pib_train.kn"))
if not english_candidates or not kannada_candidates:
    raise FileNotFoundError("Expected pib_train.en and pib_train.kn inside the ZIP.")

with open(english_candidates[0], encoding="utf-8") as f:
    english = [x.strip() for x in f]
with open(kannada_candidates[0], encoding="utf-8") as f:
    kannada = [x.strip() for x in f]

assert len(english) == len(kannada), "Parallel files have different line counts."
df = pd.DataFrame({"english": english, "kannada": kannada})
print("Loaded aligned line pairs:", len(df))
display(df.head())

## 3. Clean and split the data

In [ ]:
import unicodedata

def clean_text(x):
    return re.sub(r"\\s+", " ", unicodedata.normalize("NFC", str(x))).strip()

df["english"] = df["english"].map(clean_text)
df["kannada"] = df["kannada"].map(clean_text)
df = df[(df.english != "") & (df.kannada != "")].copy()

# Heuristic filters only; they do NOT guarantee semantic alignment.
en_words = df.english.str.split().str.len()
kn_words = df.kannada.str.split().str.len()
ratio = df.kannada.str.len().clip(lower=1) / df.english.str.len().clip(lower=1)
foreign_script = df.kannada.map(
    lambda s: any(
        ("DEVANAGARI" in unicodedata.name(c, "") or
         "BENGALI" in unicodedata.name(c, "") or
         "TAMIL" in unicodedata.name(c, "") or
         "TELUGU" in unicodedata.name(c, ""))
        for c in s
    )
)
keep = (en_words <= 100) & (kn_words <= 120) & ratio.between(0.30, 3.00) & (~foreign_script)
before = len(df)
df = df.loc[keep].drop_duplicates(["english", "kannada"]).reset_index(drop=True)
print("Rows before heuristic filtering:", before)
print("Rows after filtering:", len(df))
print("Removed:", before-len(df))

# Group by English text to avoid repeated-source leakage across splits.
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df, groups=df["english"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
temp_df = df.iloc[temp_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(temp_df, groups=temp_df["english"]))
val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

for name, part in [("train",train_df),("validation",val_df),("test",test_df)]:
    part.to_csv(os.path.join(PROJECT_DIR, f"data/processed/{name}.csv"), index=False, encoding="utf-8-sig")
print("Train/validation/test:", len(train_df), len(val_df), len(test_df))
print("Source overlap:", len(set(train_df.english)&set(val_df.english)) +
      len(set(train_df.english)&set(test_df.english)) +
      len(set(val_df.english)&set(test_df.english)))

## 4. Shared SentencePiece tokenizer for the from-scratch Transformer

In [ ]:
import sentencepiece as spm
tok_dir = os.path.join(PROJECT_DIR, "models/tokenizer")
os.makedirs(tok_dir, exist_ok=True)
tok_txt = os.path.join(tok_dir, "training_text.txt")
with open(tok_txt, "w", encoding="utf-8") as f:
    for row in train_df.itertuples(index=False):
        f.write(row.english + "\\n")
        f.write(row.kannada + "\\n")

model_prefix = os.path.join(tok_dir, "english_kannada")
if not os.path.exists(model_prefix + ".model"):
    spm.SentencePieceTrainer.train(
        input=tok_txt, model_prefix=model_prefix, vocab_size=16000,
        model_type="bpe", character_coverage=0.9995,
        pad_id=0, unk_id=1, bos_id=2, eos_id=3
    )
sp = spm.SentencePieceProcessor(model_file=model_prefix + ".model")
PAD_ID, BOS_ID, EOS_ID = sp.pad_id(), sp.bos_id(), sp.eos_id()
VOCAB_SIZE = sp.get_piece_size()
print("SentencePiece vocabulary:", VOCAB_SIZE, "special IDs:", PAD_ID, BOS_ID, EOS_ID)

## 5. Basic encoder–decoder Transformer (from scratch)

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ParallelDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len=128):
        self.frame = frame.reset_index(drop=True); self.sp = tokenizer; self.max_len=max_len
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        src = [BOS_ID] + self.sp.encode(r.english, out_type=int)[:self.max_len-2] + [EOS_ID]
        tgt = [BOS_ID] + self.sp.encode(r.kannada, out_type=int)[:self.max_len-2] + [EOS_ID]
        return torch.tensor(src), torch.tensor(tgt)

def collate(batch):
    return (pad_sequence([x[0] for x in batch], batch_first=True, padding_value=PAD_ID),
            pad_sequence([x[1] for x in batch], batch_first=True, padding_value=PAD_ID))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__(); self.drop=nn.Dropout(dropout)
        pos=torch.arange(max_len).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_len,d_model)
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div)
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return self.drop(x+self.pe[:,:x.size(1)])

class BasicTransformer(nn.Module):
    def __init__(self):
        super().__init__(); d=256
        self.emb=nn.Embedding(VOCAB_SIZE,d,padding_idx=PAD_ID)
        self.pos=PositionalEncoding(d)
        self.tr=nn.Transformer(d_model=d,nhead=4,num_encoder_layers=3,
            num_decoder_layers=3,dim_feedforward=1024,dropout=0.1,batch_first=True)
        self.out=nn.Linear(d,VOCAB_SIZE); self.scale=math.sqrt(d)
    def forward(self,src,tgt):
        sm=src.eq(PAD_ID); tm=tgt.eq(PAD_ID)
        causal=torch.triu(torch.ones(tgt.size(1),tgt.size(1),device=tgt.device,dtype=torch.bool),1)
        s=self.pos(self.emb(src)*self.scale); t=self.pos(self.emb(tgt)*self.scale)
        h=self.tr(s,t,tgt_mask=causal,src_key_padding_mask=sm,
                  tgt_key_padding_mask=tm,memory_key_padding_mask=sm)
        return self.out(h)

basic_path=os.path.join(PROJECT_DIR,"models/basic_transformer_best.pt")
basic_model=BasicTransformer().to(device)
print("Basic Transformer parameters:",sum(p.numel() for p in basic_model.parameters()))

In [ ]:
if RUN_BASIC_TRANSFORMER:
    tr_frame=train_df.iloc[:BASIC_TRAIN_LIMIT]
    va_frame=val_df
    tr_loader=DataLoader(ParallelDataset(tr_frame,sp),batch_size=32,shuffle=True,collate_fn=collate)
    va_loader=DataLoader(ParallelDataset(va_frame,sp),batch_size=32,shuffle=False,collate_fn=collate)
    opt=torch.optim.Adam(basic_model.parameters(),lr=1e-4,betas=(0.9,0.98),eps=1e-9)
    criterion=nn.CrossEntropyLoss(ignore_index=PAD_ID)
    best=float("inf"); train_hist=[]; val_hist=[]
    start=time.time()
    for epoch in range(BASIC_EPOCHS):
        basic_model.train(); losses=[]
        for src,tgt in tr_loader:
            src,tgt=src.to(device),tgt.to(device)
            opt.zero_grad(set_to_none=True)
            logits=basic_model(src,tgt[:,:-1])
            loss=criterion(logits.reshape(-1,VOCAB_SIZE),tgt[:,1:].reshape(-1))
            loss.backward(); nn.utils.clip_grad_norm_(basic_model.parameters(),1.0); opt.step()
            losses.append(loss.item())
        basic_model.eval(); vloss=[]
        with torch.no_grad():
            for src,tgt in va_loader:
                src,tgt=src.to(device),tgt.to(device)
                logits=basic_model(src,tgt[:,:-1])
                vloss.append(criterion(logits.reshape(-1,VOCAB_SIZE),tgt[:,1:].reshape(-1)).item())
        tl,vl=float(np.mean(losses)),float(np.mean(vloss))
        train_hist.append(tl); val_hist.append(vl)
        if vl<best:
            best=vl; torch.save({"model_state_dict":basic_model.state_dict(),"epoch":epoch+1,"val_loss":vl},basic_path)
        print(f"Epoch {epoch+1}/{BASIC_EPOCHS} train={tl:.4f} val={vl:.4f}")
    basic_train_seconds=time.time()-start
    print("Training minutes:",round(basic_train_seconds/60,2),"best validation loss:",best)
    plt.figure(figsize=(7,4)); plt.plot(train_hist,label="Train"); plt.plot(val_hist,label="Validation")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Basic Transformer loss"); plt.legend(); plt.grid()
    plt.savefig(os.path.join(PROJECT_DIR,"results/plots/basic_transformer_loss.png")); plt.show()
else:
    if os.path.exists(basic_path):
        basic_model.load_state_dict(torch.load(basic_path,map_location=device)["model_state_dict"])
        print("Loaded existing checkpoint.")
    else: print("Skipped training; no checkpoint exists.")

## 6. Basic Transformer evaluation

In [ ]:
def basic_translate(text, max_len=100):
    basic_model.eval()
    ids=[BOS_ID]+sp.encode(text,out_type=int)[:126]+[EOS_ID]
    src=torch.tensor([ids],device=device)
    out=[BOS_ID]
    with torch.no_grad():
        for _ in range(max_len):
            tgt=torch.tensor([out],device=device)
            nxt=basic_model(src,tgt)[0,-1].argmax().item()
            if nxt==EOS_ID: break
            out.append(nxt)
    return sp.decode(out[1:])

# Compare on a consistent held-out subset
eval_df=test_df.iloc[:EVAL_LIMIT].reset_index(drop=True)
basic_ckpt=torch.load(basic_path,map_location=device,weights_only=False) if os.path.exists(basic_path) else None
if basic_ckpt: basic_model.load_state_dict(basic_ckpt["model_state_dict"])
basic_predictions=[]
if basic_ckpt:
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    t0=time.time()
    for s in eval_df.english: basic_predictions.append(basic_translate(s))
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    basic_seconds=time.time()-t0
    basic_bleu=BLEU().corpus_score(basic_predictions,[eval_df.kannada.tolist()]).score
    basic_chrf=CHRF().corpus_score(basic_predictions,[eval_df.kannada.tolist()]).score
    print("Basic Transformer same-subset BLEU/chrF:",round(basic_bleu,4),round(basic_chrf,4))
    print("Inference seconds:",round(basic_seconds,2))
else:
    print("No Basic Transformer checkpoint available; train it or mark experiment unavailable.")

## 7. Fine-tune mT5-small

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
MT5_NAME="google/mt5-small"
mt5_dir=os.path.join(PROJECT_DIR,"models/mt5")
mt5_tokenizer=AutoTokenizer.from_pretrained(MT5_NAME)
mt5_model=AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME).to(device)
print("mT5 parameters:",sum(p.numel() for p in mt5_model.parameters()))

In [ ]:
class MT5Pairs(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        return "translate English to Kannada: "+r.english, r.kannada

def mt5_collate(batch):
    src,tgt=zip(*batch)
    x=mt5_tokenizer(list(src),padding=True,truncation=True,max_length=128,return_tensors="pt")
    y=mt5_tokenizer(text_target=list(tgt),padding=True,truncation=True,max_length=128,return_tensors="pt").input_ids
    y[y==mt5_tokenizer.pad_token_id]=-100
    return x.input_ids,x.attention_mask,y

mt5_train_loader=DataLoader(MT5Pairs(train_df.iloc[:MT5_TRAIN_LIMIT]),batch_size=8,shuffle=True,collate_fn=mt5_collate)
mt5_val_loader=DataLoader(MT5Pairs(val_df.iloc[:500]),batch_size=8,shuffle=False,collate_fn=mt5_collate)
mt5_optimizer=torch.optim.AdamW(mt5_model.parameters(),lr=5e-5,eps=1e-6)
print("mT5 batches:",len(mt5_train_loader),"validation batches:",len(mt5_val_loader))

In [ ]:
if RUN_MT5_FINETUNING:
    mt5_model.train(); mt5_losses=[]; t0=time.time()
    use_bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    for step,(ids,mask,labels) in enumerate(mt5_train_loader):
        ids,mask,labels=ids.to(device),mask.to(device),labels.to(device)
        mt5_optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.bfloat16,enabled=use_bf16):
            out=mt5_model(input_ids=ids,attention_mask=mask,labels=labels)
            loss=out.loss
        if not torch.isfinite(loss):
            print("Non-finite loss at step",step,"— stopping."); break
        loss.backward(); nn.utils.clip_grad_norm_(mt5_model.parameters(),1.0)
        mt5_optimizer.step(); mt5_losses.append(loss.item())
        if (step+1)%100==0: print(step+1,"/",len(mt5_train_loader),"loss",round(loss.item(),4))
    mt5_train_seconds=time.time()-t0
    mt5_model.eval(); vals=[]
    with torch.no_grad():
        for ids,mask,labels in mt5_val_loader:
            with torch.autocast("cuda",dtype=torch.bfloat16,enabled=use_bf16):
                vals.append(mt5_model(input_ids=ids.to(device),attention_mask=mask.to(device),labels=labels.to(device)).loss.item())
    mt5_val_loss=float(np.mean(vals))
    mt5_model.save_pretrained(mt5_dir); mt5_tokenizer.save_pretrained(mt5_dir)
    print("mT5 training minutes:",round(mt5_train_seconds/60,2),
          "mean train loss:",round(float(np.mean(mt5_losses)),4),
          "validation loss:",round(mt5_val_loss,4))
else:
    print("mT5 fine-tuning skipped.")

## 8. mT5 evaluation and temperature analysis

In [ ]:
def mt5_generate(sentences,temperature=0.0):
    preds=[]
    for i in range(0,len(sentences),8):
        batch=["translate English to Kannada: "+x for x in sentences[i:i+8]]
        inputs=mt5_tokenizer(batch,padding=True,truncation=True,max_length=128,return_tensors="pt").to(device)
        kwargs={"max_new_tokens":100,"num_beams":1}
        if temperature>0:
            kwargs.update({"do_sample":True,"temperature":temperature,"top_p":0.9})
        else:
            kwargs.update({"do_sample":False})
        with torch.no_grad():
            out=mt5_model.generate(**inputs,**kwargs)
        preds.extend(mt5_tokenizer.batch_decode(out,skip_special_tokens=True))
    return preds

if os.path.isdir(mt5_dir):
    # Evaluate deterministically on the same subset.
    mt5_model.eval()
    t0=time.time()
    mt5_predictions=mt5_generate(eval_df.english.tolist(),0.0)
    mt5_seconds=time.time()-t0
    mt5_bleu=BLEU().corpus_score(mt5_predictions,[eval_df.kannada.tolist()]).score
    mt5_chrf=CHRF().corpus_score(mt5_predictions,[eval_df.kannada.tolist()]).score
    print("mT5 same-subset BLEU/chrF:",round(mt5_bleu,4),round(mt5_chrf,4))
    print("Inference seconds:",round(mt5_seconds,2))
    # Temperature sampling on 20 examples; report metrics, do not assume higher is better.
    temp_rows=[]
    temp_frame=eval_df.iloc[:20]
    for temp in [0.2,0.7,1.0]:
        p=mt5_generate(temp_frame.english.tolist(),temp)
        b=BLEU().corpus_score(p,[temp_frame.kannada.tolist()]).score
        c=CHRF().corpus_score(p,[temp_frame.kannada.tolist()]).score
        temp_rows.append({"temperature":temp,"BLEU":b,"chrF":c})
    temperature_df=pd.DataFrame(temp_rows)
    display(temperature_df)
    temperature_df.to_csv(os.path.join(PROJECT_DIR,"results/temperature_analysis.csv"),index=False)
else:
    print("mT5 checkpoint missing; run fine-tuning cell first.")

## 9. Decoder-only LLM prompting (optional; gated models may require HF_TOKEN)

In [ ]:
# Hugging Face token is optional for public models, but required for gated repositories.
# If needed, create a read token on Hugging Face and enter it securely here.
from huggingface_hub import login
from getpass import getpass
HF_TOKEN=os.environ.get("HF_TOKEN","")
if not HF_TOKEN:
    print("If a model is gated, enter your Hugging Face token below; otherwise leave blank.")
    # HF_TOKEN=getpass("HF token (input hidden): ")
if HF_TOKEN:
    login(token=HF_TOKEN)

In [ ]:
# LLaMA 3.2 1B Instruct. Kannada is not among its officially listed supported languages.
# 4-bit loading reduces GPU memory. Access may require accepting Meta's model terms.
LLAMA_ID="meta-llama/Llama-3.2-1B-Instruct"
def load_causal_4bit(model_id):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    q=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
    tok=AutoTokenizer.from_pretrained(model_id,token=HF_TOKEN or None)
    mdl=AutoModelForCausalLM.from_pretrained(
        model_id,quantization_config=q,device_map="auto",token=HF_TOKEN or None)
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    return tok,mdl

llama_tokenizer=None; llama_model=None
if RUN_LLAMA_PROMPTING:
    try:
        llama_tokenizer,llama_model=load_causal_4bit(LLAMA_ID)
        print("LLaMA loaded.")
    except Exception as e:
        print("LLaMA could not be loaded:",repr(e))

In [ ]:
def causal_translate(tok,mdl,sentences,batch_size=2,max_new_tokens=100):
    outputs=[]; mdl.eval()
    for i in range(0,len(sentences),batch_size):
        batch=sentences[i:i+batch_size]
        prompts=[f"Translate the following English text into natural Kannada written in Kannada script. Return only the translation.\\nEnglish: {s}\\nKannada:" for s in batch]
        messages=[[{"role":"user","content":p}] for p in prompts]
        if hasattr(tok,"apply_chat_template"):
            texts=[tok.apply_chat_template(m,tokenize=False,add_generation_prompt=True) for m in messages]
        else: texts=prompts
        x=tok(texts,return_tensors="pt",padding=True,truncation=True).to(mdl.device)
        with torch.no_grad():
            y=mdl.generate(**x,max_new_tokens=max_new_tokens,do_sample=False,
                           pad_token_id=tok.pad_token_id)
        for j in range(len(batch)):
            gen=y[j,x.input_ids.shape[1]:]
            outputs.append(tok.decode(gen,skip_special_tokens=True).strip())
    return outputs

llama_predictions=None
if llama_model is not None:
    small_eval=eval_df.iloc[:LLM_EVAL_LIMIT]
    t0=time.time()
    llama_predictions=causal_translate(llama_tokenizer,llama_model,small_eval.english.tolist())
    llama_seconds=time.time()-t0
    llama_bleu=BLEU().corpus_score(llama_predictions,[small_eval.kannada.tolist()]).score
    llama_chrf=CHRF().corpus_score(llama_predictions,[small_eval.kannada.tolist()]).score
    print("LLaMA BLEU/chrF:",llama_bleu,llama_chrf,"seconds:",llama_seconds)
    del llama_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# Gemma 3 1B IT. Check model card/terms and access before running.
GEMMA_ID="google/gemma-3-1b-it"
gemma_tokenizer=None; gemma_model=None
if RUN_GEMMA_PROMPTING:
    try:
        gemma_tokenizer,gemma_model=load_causal_4bit(GEMMA_ID)
        print("Gemma loaded.")
    except Exception as e:
        print("Gemma could not be loaded:",repr(e))

gemma_predictions=None
if gemma_model is not None:
    small_eval=eval_df.iloc[:LLM_EVAL_LIMIT]
    t0=time.time()
    gemma_predictions=causal_translate(gemma_tokenizer,gemma_model,small_eval.english.tolist())
    gemma_seconds=time.time()-t0
    gemma_bleu=BLEU().corpus_score(gemma_predictions,[small_eval.kannada.tolist()]).score
    gemma_chrf=CHRF().corpus_score(gemma_predictions,[small_eval.kannada.tolist()]).score
    print("Gemma BLEU/chrF:",gemma_bleu,gemma_chrf,"seconds:",gemma_seconds)
    del gemma_model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 10. Consolidate metrics and save outputs

In [ ]:
rows=[]
def add_result(name,params,bleu,chrf,train_seconds,infer_seconds,note):
    rows.append({"Model":name,"Parameters":params,"BLEU":bleu,"chrF":chrf,
                 "Training_seconds":train_seconds,"Inference_seconds":infer_seconds,"Notes":note})

if 'basic_bleu' in globals():
    add_result("Basic Transformer",sum(p.numel() for p in basic_model.parameters()),
               basic_bleu,basic_chrf,globals().get("basic_train_seconds",np.nan),basic_seconds,
               "Trained from scratch; evaluate on common subset.")
if 'mt5_bleu' in globals():
    add_result("mT5-small",sum(p.numel() for p in mt5_model.parameters()),
               mt5_bleu,mt5_chrf,globals().get("mt5_train_seconds",np.nan),mt5_seconds,
               "Fine-tuned on subset; multilingual encoder-decoder.")
if llama_predictions is not None:
    add_result("LLaMA 3.2 1B Instruct",1230000000,llama_bleu,llama_chrf,np.nan,llama_seconds,
               "Zero-shot prompting; Kannada not officially listed as supported.")
if gemma_predictions is not None:
    add_result("Gemma 3 1B IT",1000000000,gemma_bleu,gemma_chrf,np.nan,gemma_seconds,
               "Zero-shot prompting; parameter count approximate/model-card dependent.")

results_df=pd.DataFrame(rows)
display(results_df)
results_df.to_csv(os.path.join(PROJECT_DIR,"results/model_comparison.csv"),index=False)

# Save sample translations for models that ran
pred_data={"English":eval_df.english.tolist(),"Reference Kannada":eval_df.kannada.tolist()}
if 'basic_predictions' in globals() and len(basic_predictions)==len(eval_df): pred_data["Basic Transformer"]=basic_predictions
if 'mt5_predictions' in globals() and len(mt5_predictions)==len(eval_df): pred_data["mT5"]=mt5_predictions
predictions_frame=pd.DataFrame(pred_data)
predictions_frame.to_csv(os.path.join(PROJECT_DIR,"results/predictions/model_predictions.csv"),index=False,encoding="utf-8-sig")
print("Results saved under:",os.path.join(PROJECT_DIR,"results"))

## 11. Visualizations

In [ ]:
if not results_df.empty:
    for metric in ["BLEU","chrF"]:
        vals=results_df.dropna(subset=[metric])
        plt.figure(figsize=(8,4)); plt.bar(vals.Model,vals[metric])
        plt.ylabel(metric); plt.title(f"Model comparison — {metric}")
        plt.xticks(rotation=20,ha="right"); plt.tight_layout()
        plt.savefig(os.path.join(PROJECT_DIR,f"results/plots/{metric.lower()}_comparison.png"))
        plt.show()

    vals=results_df.dropna(subset=["Inference_seconds"])
    if not vals.empty:
        plt.figure(figsize=(8,4)); plt.bar(vals.Model,vals.Inference_seconds)
        plt.ylabel("Seconds"); plt.title("Inference time on evaluated subset")
        plt.xticks(rotation=20,ha="right"); plt.tight_layout()
        plt.savefig(os.path.join(PROJECT_DIR,"results/plots/inference_time.png")); plt.show()

## 12. Ethical analysis and limitations

### Ethical considerations
- **Bias and representation:** The corpus is drawn from government/PIB material, so its vocabulary, register and topics may not represent everyday Kannada or all Kannada-speaking communities.
- **Hallucination and omission:** A fluent translation can still change names, dates, quantities, policy details or causal relationships. Human review is essential for consequential government information.
- **Cultural and linguistic representation:** Kannada has regional, social and register variation. A single reference translation cannot capture every valid expression. Legitimate loanwords and code-mixing should not automatically be treated as errors.
- **Sensitive content:** Public-sector text may discuss health, identity, conflict or vulnerable groups. Translation systems should preserve context and avoid adding unsupported claims.
- **Dataset noise:** The source corpus contains occasional alignment errors, malformed text and noisy references. Heuristic cleaning cannot detect every semantic mismatch.
- **Model access and language coverage:** Model language-support claims differ. LLaMA 3.2 1B's Kannada output is an exploratory unsupported-language probe; do not interpret it as a balanced architecture comparison.

### Limitations
- The dataset is domain-specific and not representative of general conversational translation.
- Training budgets and data subsets differ across models; report these beside every score.
- BLEU and chrF are reference-overlap metrics, not complete measures of adequacy, fluency or cultural appropriateness.
- Zero-shot prompting and fine-tuning are different adaptation settings and should not be conflated.

## 13. Generate README and experiment summary

In [ ]:
readme = """# English-to-Kannada Neural Machine Translation

Comparative evaluation of a from-scratch encoder-decoder Transformer, fine-tuned mT5-small, and prompted decoder-only LLMs on the Anuvaad PIB English-Kannada corpus (2017-2020).

## Dataset
Anuvaad Parallel Corpus: https://github.com/project-anuvaad/anuvaad-parallel-corpus
Domain: government/news text. The corpus contains some noisy or misaligned pairs.

## Models
- Basic Transformer: PyTorch encoder-decoder trained from scratch.
- mT5-small: multilingual encoder-decoder fine-tuned on a training subset.
- LLaMA 3.2 1B Instruct: zero-shot prompting; Kannada is not officially listed among its supported languages.
- Gemma 3 1B IT: zero-shot prompting; access may require accepting model terms.

## Metrics
BLEU and chrF are computed on a held-out subset. Training and inference times are recorded when experiments run successfully. Check `results/model_comparison.csv` for actual results.

## Reproducibility
Run `English_Kannada_MT_Complete.ipynb` in Google Colab. Select a GPU runtime where available, upload the dataset ZIP, and run cells in order. Checkpoints and outputs are saved to Google Drive.

## Ethical considerations and limitations
The corpus is government/news-domain specific. Automatic metrics cannot fully assess adequacy, fluency, cultural representation or hallucination. Human review is needed, especially for names, numbers and policy information. Model results must be interpreted in light of differing data budgets and language coverage.
"""
with open(os.path.join(PROJECT_DIR, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme)
print(readme)

## 14. Final file inventory

In [ ]:
for root,dirs,files_ in os.walk(PROJECT_DIR):
    level=root.replace(PROJECT_DIR,"").count(os.sep)
    if level<=3:
        print("  "*level+os.path.basename(root)+"/")
        for name in files_[:15]:
            print("  "*(level+1)+name)
print("\\nREADME:",os.path.join(PROJECT_DIR,"README.md"))
print("Comparison CSV:",os.path.join(PROJECT_DIR,"results/model_comparison.csv"))

## 15. Package outputs for download

Kaggle stores notebook outputs in `/kaggle/working`. This cell creates a ZIP containing the project data, checkpoints, metrics, plots and README. Download it from the notebook's Output/Files panel after the cell completes.

For very large model checkpoints, the ZIP may be several GB. You may omit `models/` from the archive if you only need the report/results and already have model checkpoints saved separately.

In [ ]:
import shutil, os
archive_base = "/kaggle/working/English_Kannada_MT_submission"
# Include results, processed data, README and plots. Exclude bulky checkpoints by default.
include_dir = os.path.join(PROJECT_DIR, "submission_bundle")
os.makedirs(include_dir, exist_ok=True)
for item in ["data/processed", "results", "README.md"]:
    src_item = os.path.join(PROJECT_DIR, item)
    if os.path.exists(src_item):
        dst_item = os.path.join(include_dir, item)
        if os.path.isdir(src_item):
            if os.path.exists(dst_item): shutil.rmtree(dst_item)
            shutil.copytree(src_item, dst_item)
        else:
            os.makedirs(os.path.dirname(dst_item), exist_ok=True)
            shutil.copy2(src_item, dst_item)
archive_path = shutil.make_archive(archive_base, "zip", include_dir)
print("Submission bundle:", archive_path)
print("Project root:", PROJECT_DIR)
print("Remember: Kaggle /kaggle/working files are available for download after Save Version.")